# Solar Filament Segmentation Challenge 2026 — reproducing the submission

This notebook regenerates the submitted predictions from the trained weights stored in this
repository. Nothing is retrained.

| step | what happens | where |
|---|---|---|
| 1 | fit the solar disk and the centre-to-limb intensity profile of every test image | `filament/disk.py` |
| 2 | predict a probability map with each of the 5 fold models (sliding window + 8-fold TTA) and average them | `filament/model.py`, `filament/infer.py` |
| 3 | split the map into candidate regions, score each candidate and keep those that raise the expected PQ | `filament/decision_layer.py` |
| 4 | write `output/submission.csv` and compare it with the submitted file | |

**Requirements.** The competition data, a GPU (about one hour on a single T4; CPU works but takes
many hours) and the packages in `requirements.txt`. Run the notebook from the repository root.

Training (`python -m filament.train`) and fitting the decision layer
(`python -m filament.decision_layer fit`) are documented in the README; both are optional here.

In [ ]:
import json, os, sys, time
import numpy as np
import matplotlib.pyplot as plt
import torch, sklearn

assert os.path.isdir("filament"), "run this notebook from the repository root"
sys.path.insert(0, os.getcwd())
from filament import config as C

# ---- configuration ---------------------------------------------------------
# Folder that contains `train/` and `test/` of the competition data.
DATA_ROOT = os.environ.get(
    "FILAMENT_DATA_ROOT",
    "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026")
TTA = C.TTA      # 8 = setting used for the submission
LIMIT = 0        # >0: only the first N test images (quick smoke test; output will not match)
REFIT_DECISION_LAYER = False   # True: refit the candidate scorer instead of loading models/scorer.pkl
REFERENCE = "submission/submission_dl_match.csv"   # the file submitted to Kaggle
# ----------------------------------------------------------------------------

paths = C.Paths(DATA_ROOT, work_dir="output", cache_dir="cache", weights_dir="weights",
                disk_cache="data/disk_cache_v2.json")
paths.makedirs()
assert os.path.isdir(paths.test_images), f"competition data not found under {DATA_ROOT}"
missing = [paths.weights(k) for k in range(C.N_FOLDS) if not os.path.exists(paths.weights(k))]
assert not missing, f"missing weights: {missing}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| torch", torch.__version__, "| scikit-learn", sklearn.__version__)
print(len(C.list_images(paths.test_images)), "test images")

## 1. Disk geometry and limb darkening

Filaments are detected as structures darker than their surroundings, but the quiet-Sun level
itself drops by 30–40 % from disk centre to limb. Each image is therefore divided by its own
radial median profile, which requires the disk centre and radius. The limb is located as the
sharp rise into the thin bright ring at the disk edge (Hough transform for a coarse fit, then a
robust circle fit to 720 edge points). The fitted parameters of all competition images are
shipped in `data/disk_cache_v2.json`; entries that are missing are computed here.

In [ ]:
from filament.disk import build_disk_cache, check_radii, read_gray, detect_disk
from filament.data import make_planes

disk = build_disk_cache([paths.test_images], paths.disk_cache)
test_names = C.list_images(paths.test_images)
mean_r, std_r = check_radii(disk, test_names)
print(f"disk radius of the test images: {mean_r:.1f} +/- {std_r:.1f} px")

example = test_names[0]
img = read_gray(os.path.join(paths.test_images, example))
meta = disk[example]
refit = detect_disk(img)       # recomputed from the image, to show that the cache is reproducible
print(f"{example}: cached R = {meta['R']:.2f} px, refitted R = {refit['R']:.2f} px")

planes, valid = make_planes(img, meta)
fig, ax = plt.subplots(1, 3, figsize=(16, 5.2))
ax[0].imshow(img, cmap="gray"); ax[0].set_title("H-alpha image and fitted limb")
t = np.linspace(0, 2 * np.pi, 720)
ax[0].plot(meta["cx"] + meta["R"] * np.cos(t), meta["cy"] + meta["R"] * np.sin(t), "c-", lw=0.6)
ax[1].plot(np.linspace(0, 1, C.NBINS), meta["prof"]); ax[1].set_xlabel("r / R")
ax[1].set_ylabel("median intensity"); ax[1].set_title("centre-to-limb profile")
ax[2].imshow(planes[0], cmap="gray", vmin=-1.5, vmax=1); ax[2].set_title("limb-corrected contrast (input plane 0)")
for a in (ax[0], ax[2]): a.axis("off")
plt.tight_layout(); plt.show()

## 2. Probability maps from the five fold models

The network is a U-Net with a ResNet-34 encoder whose decoder runs to full resolution. Its input
has three planes: limb-corrected contrast, raw intensity and normalised radius r/R. It was
trained with one sample per annotator reading, so its output estimates the probability that a
randomly chosen annotator marks a pixel. Images are processed at native resolution with a
512-px sliding window (stride 256), raised-cosine blending and the 8 symmetries of the square
as test-time augmentation. The weights are loaded from `weights/unet_fold{k}.pt`.

In [ ]:
from filament.infer import run_fold, set_deterministic

set_deterministic()
t0 = time.time()
wanted = test_names[:LIMIT] if LIMIT else test_names
for fold in range(C.N_FOLDS):
    out = paths.prob("test", fold)
    if os.path.exists(out) and sorted(np.load(out).files) == wanted:
        print(f"fold {fold}: {out} is complete, skipping")     # delete output/ to recompute
        continue
    run_fold(paths, fold, splits=("test",), tta=TTA, limit=LIMIT, device=device)
print(f"inference finished in {(time.time() - t0) / 60:.1f} min")

## 3. Decision layer

Thresholding the averaged probability map at 0.3 gives candidate regions. For each candidate,
21 features (probability statistics, shape, position on the disk, image contrast, context) are
fed to two gradient-boosted models that estimate *h*, the probability that an annotator drew a
matching filament, and *m*, the expected IoU of that match. Since
PQ = Σ IoU / (TP + ½FP + ½FN), emitting a candidate raises PQ exactly when *h·m* > PQ/2; the
threshold found on out-of-fold data agrees with this value.

The scorer was fitted on out-of-fold candidates of the training images and is loaded from
`models/scorer.pkl`. Set `REFIT_DECISION_LAYER = True` above to refit it instead; this first
predicts all training images with the fold model that did not see them (several GPU hours)
unless `output/val_prob_fold*.npz` already exist.

In [ ]:
from filament import decision_layer as dl

if REFIT_DECISION_LAYER:
    for fold in range(C.N_FOLDS):
        if not os.path.exists(paths.prob("val", fold)):
            run_fold(paths, fold, splits=("val",), tta=TTA, device=device)
    build_disk_cache([paths.train_images], paths.disk_cache)
    dl.fit(paths, "models")

with open("models/decision_layer_params.json") as f:
    params = json.load(f)
print(f"out-of-fold PQ: hysteresis baseline {params['baseline_pq']:.4f} -> decision layer "
      f"{params['oof_pq']:.4f} (95% CI of the gain [{params['delta_ci95'][0]:+.4f}, "
      f"{params['delta_ci95'][1]:+.4f}]); tau = {params['tau']}, PQ/2 = {params['oof_pq'] / 2:.3f}")

submission = dl.predict(paths, "models", "output/submission.csv")

In [ ]:
# The three stages on one test image: averaged probability, candidates, emitted instances.
archives = [np.load(paths.prob("test", k)) for k in range(C.N_FOLDS)]
prob = dl.ensemble_prob(archives, example)
valid = dl.disk_grid(meta)[2] < C.DISK_FRAC
cand, n_cand = dl.label_candidates(prob, valid, params["mask_t"])
final = dl.decode_submission("output/submission.csv").get(os.path.splitext(example)[0], [])
print(f"{example}: {n_cand} candidates -> {len(final)} emitted instances")

ys, xs = np.nonzero(cand) if n_cand else (np.array([1024]), np.array([1024]))
y0 = int(np.clip(np.median(ys) - 384, 0, C.FULL - 768)); x0 = int(np.clip(np.median(xs) - 384, 0, C.FULL - 768))
win = (slice(y0, y0 + 768), slice(x0, x0 + 768))
inst = np.zeros((C.FULL, C.FULL), np.int32)
for i, m in enumerate(final, 1):
    inst[m] = i
fig, ax = plt.subplots(1, 4, figsize=(18, 4.8))
ax[0].imshow(img[win], cmap="gray"); ax[0].set_title("image (768-px window)")
ax[1].imshow(prob[win], cmap="magma", vmin=0, vmax=255); ax[1].set_title("ensemble probability")
ax[2].imshow(img[win], cmap="gray"); ax[2].imshow(np.ma.masked_equal(cand[win], 0), cmap="tab20", alpha=0.7, interpolation="nearest")
ax[2].set_title(f"candidates (p > {params['mask_t']})")
ax[3].imshow(img[win], cmap="gray"); ax[3].imshow(np.ma.masked_equal(inst[win], 0), cmap="tab20", alpha=0.7, interpolation="nearest")
ax[3].set_title("emitted instances")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 4. Comparison with the submitted file

GPU convolutions are not guaranteed to be bit-identical across hardware and library versions,
so a handful of boundary pixels may differ from the submitted file. The summary below counts
instances that are pixel-identical and the total number of pixels that differ.

In [ ]:
if os.path.exists(REFERENCE):
    cmp = dl.compare_submissions("output/submission.csv", REFERENCE)
    print(json.dumps(cmp, indent=2))
    if cmp["byte_identical"]:
        print("The reproduced submission is byte-identical to the submitted file.")
    else:
        print(f"{cmp['identical_instances']} of {cmp['instances_b']} submitted instances reproduced "
              f"pixel-for-pixel; {cmp['differing_pixels']} pixels differ in total.")
else:
    print("reference submission not found:", REFERENCE)